In [1]:
import os
from pathlib import Path
import subprocess
import rasterio
import matplotlib.pyplot as plt

# ==== USER CONFIGURATION ====

BASE_URL = "https://gws-access.jasmin.ac.uk/public/nceo_geohazards/LiCSAR_products/62/062D_05831_131313/"
CUTOFF = "20170624"
MAX_DAYS = 72

LICSAR_DIR = Path("./licsar_pre_event").resolve()
MINTPY_DIR = Path("./mintpy_project").resolve()

# Bounding box (west, south, east, north)
BBOX = (103.65, 32.06, 103.68, 32.10)

# Interferogram covering the landslide
LANDSLIDE_PAIR

NameError: name 'LANDSLIDE_PAIR' is not defined

In [2]:
import argparse
import concurrent.futures as futures
import re
import sys
import time
from dataclasses import dataclass
from datetime import datetime
from html.parser import HTMLParser
from urllib.parse import urljoin, urlparse, unquote
import requests
from pathlib import Path
from typing import Iterable

PAIR_RE = re.compile(r"^(\d{8})_(\d{8})/?$")
FRAME_RE = re.compile(r"LiCSAR_products/([^/]+)/([^/]+)/?")
URL_RE = re.compile(r"https?://[^\s\"'<>]+")


In [3]:
import argparse
import concurrent.futures as futures
import re
import sys
import time
from dataclasses import dataclass
from datetime import datetime
from html.parser import HTMLParser
from urllib.parse import urljoin, urlparse, unquote
import requests
from pathlib import Path
from typing import Iterable

PAIR_RE = re.compile(r"^(\d{8})_(\d{8})/?$")
FRAME_RE = re.compile(r"LiCSAR_products/([^/]+)/([^/]+)/?")
URL_RE = re.compile(r"https?://[^\s\"'<>]+")


In [4]:
class LinkParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []

    def handle_starttag(self, tag, attrs):
        if tag.lower() != "a":
            return
        href = dict(attrs).get("href")
        if href:
            self.links.append(href)


@dataclass(frozen=True)
class IfgPair:
    name: str
    start: datetime
    end: datetime
    listing_url: str

    @property
    def temporal_baseline_days(self):
        return (self.end - self.start).days


def make_session():
    s = requests.Session()
    s.headers.update({"User-Agent": "Mozilla/5.0 LiCSAR-student-downloader"})
    return s


def get_text(session, url):
    r = session.get(url, timeout=60, allow_redirects=True)
    r.raise_for_status()
    return r.text


In [5]:
def list_hrefs(session, url):
    text = get_text(session, url)
    parser = LinkParser()
    parser.feed(text)

    links = []
    links.extend(parser.links)
    links.extend(URL_RE.findall(text))

    bare = re.findall(
        r"\b(?:\d{8}_\d{8}/?|[A-Za-z0-9_.-]+\.(?:tif|png|txt|csv)|baselines|network\.png)\b",
        text,
    )
    links.extend(bare)

    seen = set()
    clean = []
    for link in links:
        link = link.strip()
        if link and link not in seen:
            clean.append(link)
            seen.add(link)
    return clean


def parse_date(s):
    return datetime.strptime(s, "%Y%m%d")


def derive_ceda_product_root(base_url):
    m = FRAME_RE.search(base_url)
    track, frame = m.groups()
    return f"https://data.ceda.ac.uk/neodc/comet/data/licsar_products/{track}/{frame}/"


In [6]:
def filename_from_href_or_url(href):
    parsed = urlparse(href)
    if parsed.scheme and parsed.netloc:
        return unquote(Path(parsed.path).name)
    return unquote(Path(href.strip("/")).name)


def is_absolute_url(href):
    parsed = urlparse(href)
    return bool(parsed.scheme and parsed.netloc)


def resolve_file_url(href, listing_url, ceda_root, pair_name=None, metadata=False):
    filename = filename_from_href_or_url(href)

    if is_absolute_url(href):
        return href, filename

    if pair_name:
        return urljoin(ceda_root, f"{pair_name}/{filename}"), filename

    if metadata:
        return urljoin(ceda_root, f"metadata/{filename}"), filename

    return urljoin(listing_url.rstrip("/") + "/", href), filename


In [7]:
def find_ifg_pairs(session, base_url, cutoff, max_days, include_cutoff=False):
    ifg_listing = urljoin(base_url.rstrip("/") + "/", "interferograms/")
    cutoff_dt = parse_date(cutoff)

    hrefs = list_hrefs(session, ifg_listing)
    pairs = []

    for href in hrefs:
        name = filename_from_href_or_url(href)
        m = PAIR_RE.match(name)
        if not m:
            continue

        d1, d2 = m.groups()
        start, end = parse_date(d1), parse_date(d2)

        if include_cutoff:
            if end > cutoff_dt:
                continue
        else:
            if end >= cutoff_dt:
                continue

        if max_days and (end - start).days > max_days:
            continue

        listing = urljoin(ifg_listing.rstrip("/") + "/", name)
        pairs.append(IfgPair(name, start, end, listing))

    return sorted(pairs, key=lambda p: (p.start, p.end))


In [8]:
def should_download_file(filename, products):
    if "all" in products:
        return True
    if "unw" in products and filename.endswith(".geo.unw.tif"):
        return True
    if "cc" in products and filename.endswith(".geo.cc.tif"):
        return True
    if "png" in products and filename.endswith(".png"):
        return True
    return False


def download_file(session, url, out_path, overwrite=False, retries=3):
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists() and not overwrite:
        return out_path, "exists", url

    tmp = out_path.with_suffix(out_path.suffix + ".part")

    for attempt in range(1, retries + 1):
        try:
            with session.get(url, stream=True, timeout=180) as r:
                r.raise_for_status()
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)
            tmp.replace(out_path)
            return out_path, "downloaded", url
        except Exception as e:
            if tmp.exists():
                tmp.unlink()
            time.sleep(2 * attempt)

    raise RuntimeError(f"Failed to download {url}")


In [9]:
def download_ifg_pair(session, pair, ceda_root, out_dir, products, overwrite):
    hrefs = list_hrefs(session, pair.listing_url)
    selected = []

    for href in hrefs:
        url, filename = resolve_file_url(href, pair.listing_url, ceda_root, pair_name=pair.name)
        if should_download_file(filename, products):
            selected.append((url, out_dir / "interferograms" / pair.name / filename))

    results = []
    for url, path in selected:
        print("Downloading:", url)
        results.append(download_file(session, url, path, overwrite))
    return results


In [10]:
def run_download():
    session = make_session()
    ceda_root = derive_ceda_product_root(BASE_URL)

    pairs = find_ifg_pairs(session, BASE_URL, CUTOFF, MAX_DAYS, include_cutoff=False)
    print(f"Found {len(pairs)} interferograms")

    download_metadata(session, BASE_URL, ceda_root, LICSAR_DIR, overwrite=False, workers=4)

    for pair in pairs:
        download_ifg_pair(session, pair, ceda_root, LICSAR_DIR, {"unw", "cc", "png"}, overwrite=False)

run_download()


Found 104 interferograms


NameError: name 'download_metadata' is not defined

In [11]:
import h5py
import numpy as np
import rasterio
from rasterio.windows import from_bounds, Window
import re

PAIR_RE = re.compile(r"^(\d{8})_(\d{8})$")
S1_WAVELENGTH_M = 0.05546576


ModuleNotFoundError: No module named 'h5py'

In [12]:
def find_pairs(licsar_dir):
    ifg_dir = licsar_dir / "interferograms"
    pairs = []

    for pair_dir in sorted(ifg_dir.iterdir()):
        if not pair_dir.is_dir():
            continue
        pair = pair_dir.name
        if not PAIR_RE.match(pair):
            continue

        unw = pair_dir / f"{pair}.geo.unw.tif"
        cc = pair_dir / f"{pair}.geo.cc.tif"
        if unw.exists() and cc.exists():
            pairs.append((pair, unw, cc))

    return pairs


In [13]:
def read_baselines(metadata_dir):
    baseline_file = metadata_dir / "baselines"
    if not baseline_file.exists():
        return {}

    out = {}
    with open(baseline_file, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if not line.strip() or line.startswith("#"):
                continue
            m = re.search(r"\b(20\d{6}|19\d{6})\b", line)
            if not m:
                continue
            date = m.group(1)
            nums = re.findall(r"[-+]?\d+(?:\.\d+)?", line.replace(date, " "))
            if nums:
                out[date] = float(nums[0])
    return out


In [14]:
def get_window(ds, bbox):
    if bbox is None:
        return None
    west, south, east, north = bbox
    raw = from_bounds(west, south, east, north, ds.transform)
    win = raw.round_offsets().round_lengths()
    col_off = max(0, int(win.col_off))
    row_off = max(0, int(win.row_off))
    col_end = min(ds.width, int(win.col_off + win.width))
    row_end = min(ds.height, int(win.row_off + win.height))
    return Window(col_off, row_off, col_end - col_off, row_end - row_off)


def read_raster(path, window=None, dtype=np.float32):
    with rasterio.open(path) as ds:
        arr = ds.read(1, window=window).astype(dtype)
        profile = ds.profile.copy()
        transform = ds.window_transform(window) if window else ds.transform
        nodata = ds.nodata

    if nodata is not None:
        arr[arr == nodata] = np.nan
    arr[~np.isfinite(arr)] = 0.0

    profile["transform"] = transform
    profile["height"], profile["width"] = arr.shape
    return arr, profile


NameError: name 'np' is not defined

In [15]:
def profile_to_mintpy_attrs(profile, project_name, orbit_direction):
    transform = profile["transform"]
    length = profile["height"]
    width = profile["width"]

    return {
        "PROJECT_NAME": project_name,
        "PROCESSOR": "licsar",
        "PLATFORM": "Sen",
        "SENSOR": "Sen",
        "WAVELENGTH": f"{S1_WAVELENGTH_M:.8f}",
        "ORBIT_DIRECTION": orbit_direction,
        "LENGTH": str(length),
        "WIDTH": str(width),
        "FILE_LENGTH": str(length),
        "X_FIRST": str(transform.c),
        "Y_FIRST": str(transform.f),
        "X_STEP": str(transform.a),
        "Y_STEP": str(transform.e),
        "UNIT": "radian",
    }


In [16]:
def make_geometry(licsar_dir, out_dir, profile, attrs, window=None):
    metadata_dir = licsar_dir / "metadata"
    out_file = out_dir / "inputs" / "geometryGeo.h5"
    out_file.parent.mkdir(parents=True, exist_ok=True)

    hgt_file = next(metadata_dir.glob("*geo.hgt.tif"), None)

    with h5py.File(out_file, "w") as h5:
        for k, v in attrs.items():
            h5.attrs[k] = v

        if hgt_file:
            height, _ = read_raster(hgt_file, window)
        else:
            height = np.zeros((profile["height"], profile["width"]), dtype=np.float32)

        h5.create_dataset("height", data=height)


In [17]:
def make_ifgram_stack(licsar_dir, out_dir, pairs, profile, attrs, window=None):
    metadata_dir = licsar_dir / "metadata"
    baseline_map = read_baselines(metadata_dir)

    out_file = out_dir / "inputs" / "ifgramStack.h5"
    out_file.parent.mkdir(parents=True, exist_ok=True)

    length = profile["height"]
    width = profile["width"]
    num_ifg = len(pairs)

    date_pairs = []
    bperp = np.zeros(num_ifg, dtype=np.float32)

    for i, (pair, _, _) in enumerate(pairs):
        d1, d2 = pair.split("_")
        date_pairs.append([d1.encode(), d2.encode()])
        if d1 in baseline_map and d2 in baseline_map:
            bperp[i] = baseline_map[d2] - baseline_map[d1]

    with h5py.File(out_file, "w") as h5:
        for k, v in attrs.items():
            h5.attrs[k] = v

        h5.create_dataset("date", data=np.array(date_pairs, dtype="S8"))
        h5.create_dataset("bperp", data=bperp)

        unw_ds = h5.create_dataset("unwrapPhase", (num_ifg, length, width), dtype=np.float32)
        cc_ds = h5.create_dataset("coherence", (num_ifg, length, width), dtype=np.float32)

        for i, (pair, unw_path, cc_path) in enumerate(pairs):
            unw, _ = read_raster(unw_path, window)
            cc, _ = read_raster(cc_path, window)
            cc = np.clip(cc / 255.0, 0, 1)

            unw_ds[i] = unw
            cc_ds[i] = cc


In [18]:
def run_conversion():
    pairs = find_pairs(LICSAR_DIR)
    print("Found", len(pairs), "interferograms")

    first_unw = pairs[0][1]
    with rasterio.open(first_unw) as ds:
        window = get_window(ds, BBOX)
        _, profile = read_raster(first_unw, window)

    attrs = profile_to_mintpy_attrs(profile, MINTPY_DIR.name, "descending")

    make_geometry(LICSAR_DIR, MINTPY_DIR, profile, attrs, window)
    make_ifgram_stack(LICSAR_DIR, MINTPY_DIR, pairs, profile, attrs, window)

run_conversion()


FileNotFoundError: [WinError 3] System nie może odnaleźć określonej ścieżki: 'C:\\Users\\ada\\AppData\\Local\\Programs\\Microsoft VS Code\\licsar_pre_event\\interferograms'

In [19]:
cc_path = LICSAR_DIR / "interferograms" / LANDSLIDE_PAIR / f"{LANDSLIDE_PAIR}.geo.cc.tif"

with rasterio.open(cc_path) as ds:
    cc = ds.read(1)
    extent = (ds.bounds.left, ds.bounds.right, ds.bounds.bottom, ds.bounds.top)

plt.figure(figsize=(6,5))
plt.imshow(cc, cmap="viridis", vmin=0, vmax=1, extent=extent)
plt.colorbar(label="Coherence")
plt.title(f"Coherence – {LANDSLIDE_PAIR}")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()


NameError: name 'LANDSLIDE_PAIR' is not defined